# 03. Знакомство с <span style="color:#7CDBA0">PyTorch</span> — решения

Смотри только после своей попытки. Задания — в `03_знакомство_с_pytorch.ipynb`.

In [ ]:
import torch

torch.manual_seed(0)
print(torch.__version__)


def need_tensor(x, name):
    assert isinstance(x, torch.Tensor), f"{name} — пока не тензор (замени ... на код)"

## 1. Создание <span style="color:#7CDBA0">тензоров</span>

`<span style="color:#E6C07B">torch.tensor</span>(список)` — как `np.array`. Полезные аргументы: `dtype`, <span style="color:#C9A0DC">requires_grad</span>, <span style="color:#E6C07B">device</span>.

**1.1** Создай одномерный тензор `a` из списка `[1, 2, 3, 4]`.

In [ ]:
a = torch.tensor([1, 2, 3, 4])

In [ ]:
need_tensor(a, "a")
assert a.shape == (4,), f"форма {tuple(a.shape)}, а нужна (4,)"
assert a.dtype == torch.int64, f"dtype {a.dtype}: из целых чисел по умолчанию получается int64"
print("1.1 — верно")

**1.2** Создай двумерный тензор `b` из вложенного списка `[[1, 2, 3], [4, 5, 6]]` с типом `float32`.

In [ ]:
b = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)

In [ ]:
need_tensor(b, "b")
assert b.shape == (2, 3), f"форма {tuple(b.shape)}, а нужна (2, 3)"
assert b.dtype == torch.float32, f"dtype {b.dtype}, а нужен float32"
print("1.2 — верно")

**1.3** Создай тензор `w` из `[0.5, -1.0, 2.0]`, для которого PyTorch будет считать градиенты.

In [ ]:
w = torch.tensor([0.5, -1.0, 2.0], requires_grad=True)

In [ ]:
need_tensor(w, "w")
assert w.requires_grad, "забыл requires_grad=True"
assert w.is_floating_point(), "градиенты считаются только для дробных чисел"
print("1.3 — верно")

**1.4** Автор показывал, что <span style="color:#C9A0DC">requires_grad=True</span> для целых чисел даёт ошибку. Поймай её: внутри `try` создай тензор из `[1, 2, 3]` с <span style="color:#C9A0DC">requires_grad=True</span>, а текст ошибки сохрани в `err`.

In [ ]:
err = None
try:
    torch.tensor([1, 2, 3], requires_grad=True)
except RuntimeError as e:
    err = str(e)

print(err)

In [ ]:
assert err is not None, "ошибка не возникла — проверь, что числа целые и стоит requires_grad=True"
assert "floating point" in err
print("1.4 — верно. Вывод: requires_grad только для float (и complex)")

## 2. Свойства <span style="color:#7CDBA0">тензора</span> и <span style="color:#C9A0DC">item()</span>

`dtype`, `shape` (то же, что `size()`), `ndim`.

**2.1** Дан тензор `x`. Запиши в переменные его тип, форму и число измерений, используя свойства тензора (не вписывай числа руками).

In [ ]:
x = torch.rand(2, 3, 4)

In [ ]:
x_dtype = x.dtype
x_shape = x.shape
x_ndim = x.ndim

In [ ]:
assert x_dtype == torch.float32
assert x_shape == (2, 3, 4) and x_shape == x.size(), "shape и size() дают одно и то же"
assert x_ndim == 3
print("2.1 — верно")

**2.2** Из тензора `m` достань элемент `5`:
- `el` — как тензор (через индексацию);
- `num` — как обычное число Python.

In [ ]:
m = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], requires_grad=True)

In [ ]:
el = m[1, 1]
num = m[1, 1].item()

print(el, type(el))
print(num, type(num))

In [ ]:
need_tensor(el, "el")
assert el.ndim == 0, "индексация одного элемента даёт тензор без измерений (скаляр)"
assert isinstance(num, float) and num == 5.0, "число получают методом item()"
print("2.2 — верно")

**2.3** Так `item()` используют на практике: сохраняют значение loss по эпохам. Собери в список `history` значения из `losses` как обычные числа.

In [ ]:
losses = [(w * k).sum() for k in range(1, 4)]
print(losses)

In [ ]:
history = [loss.item() for loss in losses]

In [ ]:
assert isinstance(history, list) and len(history) == 3
assert all(isinstance(v, float) for v in history), "в history должны быть float, а не тензоры"
print("2.3 — верно:", history)

> Вопрос себе: почему в истории лучше хранить числа, а не сами тензоры `losses`? Подсказка: что ещё тащит за собой тензор (смотри <span style="color:#C9A0DC">grad_fn</span> в выводе выше).

## 3. Функции создания

`zeros`, `ones`, `zeros_like`, `full`, `full_like`, `arange`, `eye`, `diag`, `tril`.

**3.1** Создай:
- `z` — нули формы (3, 4);
- `zi` — нули формы (3, 4) с типом `int32`;
- `o` — единицы формы (2, 2);
- `zl` — нули той же формы, что у `b` (не вписывай форму руками).

In [ ]:
z = torch.zeros(3, 4)
zi = torch.zeros(3, 4, dtype=torch.int32)
o = torch.ones(2, 2)
zl = torch.zeros_like(b)

In [ ]:
for t, name in [(z, "z"), (zi, "zi"), (o, "o"), (zl, "zl")]:
    need_tensor(t, name)
assert z.shape == (3, 4) and z.sum() == 0 and z.dtype == torch.float32
assert zi.dtype == torch.int32
assert o.shape == (2, 2) and o.sum() == 4
assert zl.shape == b.shape and zl.sum() == 0
print("3.1 — верно")

**3.2** Тензоры, заполненные одним числом:
- `f1` — форма (2, 3), все элементы `7.0` (<span style="color:#E6C07B">torch.full</span>, форму задаёшь сам);
- `f2` — форма как у `b`, все элементы `7.0` (<span style="color:#E6C07B">torch.full_like</span>, форму берёт из тензора).

In [ ]:
f1 = torch.full((2, 3), 7.0)
f2 = torch.full_like(b, 7.0)

In [ ]:
need_tensor(f1, "f1"); need_tensor(f2, "f2")
assert f1.shape == (2, 3) and (f1 == 7).all()
assert f2.shape == b.shape and (f2 == 7).all()
print("3.2 — верно")

**3.3**
- `r` — чётные числа от 0 до 8 включительно через <span style="color:#E6C07B">torch.arange</span>;
- `e` — единичная матрица 3×3;
- `d` — диагональная матрица с `1, 2, 3` на диагонали;
- `lt` — нижнетреугольная матрица 3×3 из единиц (подсказка: <span style="color:#E6C07B">torch.tril</span> от матрицы единиц).

In [ ]:
r = torch.arange(0, 10, 2)
e = torch.eye(3)
d = torch.diag(torch.tensor([1, 2, 3]))
lt = torch.tril(torch.ones(3, 3))

In [ ]:
for t, name in [(r, "r"), (e, "e"), (d, "d"), (lt, "lt")]:
    need_tensor(t, name)
assert r.tolist() == [0, 2, 4, 6, 8], f"получилось {r.tolist()}: правая граница в arange не входит"
assert torch.equal(e, torch.tensor([[1., 0, 0], [0, 1, 0], [0, 0, 1]]))
assert d.tolist() == [[1, 0, 0], [0, 2, 0], [0, 0, 3]]
assert lt.tolist() == [[1, 0, 0], [1, 1, 0], [1, 1, 1]]
print("3.3 — верно")
print(lt)

## 4. Изменение формы: <span style="color:#C9A0DC">view</span>, <span style="color:#C9A0DC">reshape</span>, <span style="color:#C9A0DC">unsqueeze</span>

**4.1** Дан `v = <span style="color:#E6C07B">torch.arange</span>(12)`. Получи:
- `v34` — форма (3, 4) через <span style="color:#C9A0DC">view</span>;
- `v223` — форма (2, 2, 3) через <span style="color:#C9A0DC">reshape</span>;
- `v4x` — 4 строки, а число столбцов пусть PyTorch посчитает сам (через `-1`).

In [ ]:
v = torch.arange(12)

In [ ]:
v34 = v.view(3, 4)
v223 = v.reshape(2, 2, 3)
v4x = v.view(4, -1)

In [ ]:
for t, name in [(v34, "v34"), (v223, "v223"), (v4x, "v4x")]:
    need_tensor(t, name)
assert v34.shape == (3, 4)
assert v223.shape == (2, 2, 3)
assert v4x.shape == (4, 3), "-1 значит: вычисли этот размер сам (12 / 4 = 3)"
print("4.1 — верно")
print(v34)

**4.2** `unsqueeze(dim)` добавляет ось размера 1. Дан `p` формы (3,). Получи:
- `p_row` — форма (1, 3);
- `p_col` — форма (3, 1);
- `p_back` — из `p_row` обратно форма (3,) через `squeeze`.

In [ ]:
p = torch.tensor([1.0, 2.0, 3.0])

In [ ]:
p_row = p.unsqueeze(0)
p_col = p.unsqueeze(1)
p_back = p_row.squeeze(0)

In [ ]:
for t, name in [(p_row, "p_row"), (p_col, "p_col"), (p_back, "p_back")]:
    need_tensor(t, name)
assert p_row.shape == (1, 3)
assert p_col.shape == (3, 1)
assert p_back.shape == (3,)
print("4.2 — верно")

**4.3** Автор сказал, что <span style="color:#C9A0DC">view</span> и <span style="color:#C9A0DC">reshape</span> делают одно и то же. Почти: <span style="color:#C9A0DC">view</span> работает только когда данные лежат в памяти подряд. После транспонирования (`.T`) это не так.

Дан `mt`. Внутри `try` попробуй вытянуть его в вектор через `view(6)`, текст ошибки сохрани в `err_view`. Затем получи вектор `flat` через <span style="color:#C9A0DC">reshape</span>.

In [ ]:
mt = torch.arange(6).view(2, 3).T
print(mt)

In [ ]:
err_view = None
try:
    mt.view(6)
except RuntimeError as e:
    err_view = str(e)

flat = mt.reshape(6)

In [ ]:
assert err_view is not None and "view size is not compatible" in err_view
need_tensor(flat, "flat")
assert flat.tolist() == [0, 3, 1, 4, 2, 5]
print("4.3 — верно. Правило: если не уверен — reshape")

## 5. Арифметика

**5.1** Дан `q`. Получи `q2` — `q`, умноженный на 2, и `qq` — сумму `q` с самим собой.

In [ ]:
q = torch.tensor([1.0, 2.0, 3.0])

In [ ]:
q2 = q * 2
qq = q + q

In [ ]:
need_tensor(q2, "q2"); need_tensor(qq, "qq")
assert q2.tolist() == [2.0, 4.0, 6.0] and torch.equal(q2, qq)
print("5.1 — верно")

**5.2** Сложение тензора со списком падает. Поймай тип ошибки в `err_type` (например `"ValueError"`), а потом исправь: получи `fixed` = `q` + `[10, 20, 30]`, превратив список в тензор.

In [ ]:
err_type = None
try:
    q + [10, 20, 30]
except Exception as e:
    err_type = type(e).__name__

fixed = q + torch.tensor([10, 20, 30])

In [ ]:
assert err_type == "TypeError", f"получил {err_type}"
need_tensor(fixed, "fixed")
assert fixed.tolist() == [11.0, 22.0, 33.0]
print("5.2 — верно")

## 6. <span style="color:#C9A0DC">sum</span> / <span style="color:#C9A0DC">mean</span> по осям и <span style="color:#C9A0DC">keepdim</span>

`dim` — по какой оси сворачиваем. Эта ось из формы **исчезает**, а с `keepdim=True` остаётся с размером 1.

In [ ]:
s = torch.arange(6, dtype=torch.float32).view(2, 3)
print(s)

**6.1** Посчитай:
- `s_sum` — сумма всех элементов;
- `s_mean` — среднее всех элементов;
- `col_mean` — среднее каждого столбца (форма (3,));
- `row_mean` — среднее каждой строки (форма (2,));
- `row_mean_k` — среднее каждой строки с `keepdim=True`.

Перед запуском проверки предскажи, какие получатся числа.

In [ ]:
s_sum = s.sum()
s_mean = s.mean()
col_mean = s.mean(dim=0)
row_mean = s.mean(dim=1)
row_mean_k = s.mean(dim=1, keepdim=True)

In [ ]:
for t, name in [(s_sum, "s_sum"), (s_mean, "s_mean"), (col_mean, "col_mean"),
                (row_mean, "row_mean"), (row_mean_k, "row_mean_k")]:
    need_tensor(t, name)
assert s_sum.item() == 15.0 and s_mean.item() == 2.5
assert col_mean.tolist() == [1.5, 2.5, 3.5], "dim=0 — сворачиваем строки, остаётся по числу на столбец"
assert row_mean.tolist() == [1.0, 4.0]
assert row_mean_k.shape == (2, 1), f"форма {tuple(row_mean_k.shape)}, а с keepdim нужна (2, 1)"
print("6.1 — верно")

**6.2** Зачем нужен <span style="color:#C9A0DC">keepdim</span>. Хотим из каждой строки `s` вычесть её среднее, чтобы среднее строки стало 0.

- внутри `try` вычти `row_mean` (форма (2,)) из `s` (форма (2, 3)), текст ошибки сохрани в `err_bc`;
- потом получи `centered` правильно, через `row_mean_k`.

In [ ]:
err_bc = None
try:
    s - row_mean
except RuntimeError as e:
    err_bc = str(e)

print(err_bc)
centered = s - row_mean_k

In [ ]:
assert err_bc is not None, "ошибки не было — вычитать нужно row_mean без keepdim"
need_tensor(centered, "centered")
assert centered.shape == (2, 3)
assert torch.allclose(centered.mean(dim=1), torch.zeros(2))
print("6.2 — верно")
print(centered)

> Почему так: (2, 3) и (2,) PyTorch выравнивает **справа** — 3 против 2, не совпадает. А (2, 3) и (2, 1) — ось размера 1 растягивается до 3, всё работает. Это broadcasting, как в NumPy.

**6.3** Предскажи формы **без запуска кода**: впиши кортежи руками, проверка сравнит с настоящими.

Дан `y` формы (4, 5, 6).

In [ ]:
y = torch.rand(4, 5, 6)

In [ ]:
shape_a = (4, 6)  # y.sum(dim=1)
shape_b = (1, 5, 6)  # y.mean(dim=0, keepdim=True)
shape_c = (5,)  # y.sum(dim=(0, 2))
shape_d = (4, 5)  # y.mean(dim=-1)

In [ ]:
real = {
    "shape_a": y.sum(dim=1).shape,
    "shape_b": y.mean(dim=0, keepdim=True).shape,
    "shape_c": y.sum(dim=(0, 2)).shape,
    "shape_d": y.mean(dim=-1).shape,
}
mine = {"shape_a": shape_a, "shape_b": shape_b, "shape_c": shape_c, "shape_d": shape_d}
wrong = [k for k in real if tuple(real[k]) != mine[k]]
assert not wrong, "не совпало: " + ", ".join(f"{k}: ты {mine[k]}, на деле {tuple(real[k])}" for k in wrong)
print("6.3 — верно")

## 7. <span style="color:#E6C07B">CPU</span> и <span style="color:#E6C07B">GPU</span>

У тебя PyTorch собран под CPU, поэтому <span style="color:#E6C07B">cuda</span> будет `False`, и это нормально. Важно выучить **шаблон**: код с ним работает и на CPU, и на GPU без правок.

**7.1** Запиши, доступны ли CPU и GPU, в `cpu_ok` и `cuda_ok`. Затем заведи переменную <span style="color:#E6C07B">device</span>: строка <span style="color:#E6C07B">"cuda"</span>, если GPU доступен, иначе <span style="color:#E6C07B">"cpu"</span>.

In [ ]:
cpu_ok = torch.cpu.is_available()
cuda_ok = torch.cuda.is_available()
device = "cuda" if torch.cuda.is_available() else "cpu"

print(cpu_ok, cuda_ok, device)

In [ ]:
assert cpu_ok is True
assert cuda_ok == torch.cuda.is_available()
assert device == ("cuda" if cuda_ok else "cpu")
print("7.1 — верно")

**7.2** Создай `g` из `[1.0, 2.0]` с <span style="color:#C9A0DC">requires_grad=True</span> и перенеси его на <span style="color:#E6C07B">device</span> через <span style="color:#C9A0DC">.to(...)</span>.

In [ ]:
g = torch.tensor([1.0, 2.0], requires_grad=True).to(device)

In [ ]:
need_tensor(g, "g")
assert g.device.type == device, f"g на {g.device}, а должен быть на {device}"
print("7.2 — верно, g.device =", g.device)

**7.3** Посчитай `h = g * 3` — у `h` появится <span style="color:#C9A0DC">grad_fn</span> (функция для градиентов). Получи `h_cpu` — копию `h` на CPU **без** связи с графом градиентов (`detach`, потом `cpu`).

In [ ]:
h = g * 3
print(h)

h_cpu = h.detach().cpu()
print(h_cpu)

In [ ]:
need_tensor(h_cpu, "h_cpu")
assert h.grad_fn is not None
assert h_cpu.grad_fn is None and not h_cpu.requires_grad, "нужен detach()"
assert h_cpu.device.type == "cpu"
print("7.3 — верно")

> Два уточнения к видео:
> - тензоры на разных устройствах складывать нельзя — будет `RuntimeError: Expected all tensors to be on the same device`. Проверить у себя не выйдет без GPU, просто запомни;
> - копия создаётся, только когда тензор реально переезжает. Если он уже на нужном устройстве, <span style="color:#C9A0DC">.to(...)</span> возвращает **тот же** тензор: проверь `g.to("cpu") is g` на CPU.

## 8. Разминка на формы

В видео этого почти нет, но в нейросетях это главное. Формы пиши в комментариях рядом с кодом: `# (B, C, H, W)`.

**8.1** Рекомендация одному пользователю. Даны `history` — ID 5 последних товаров `(T,)` и `history_emb` — их эмбеддинги `(T, E)` = (5, 4). Получи `history_batch` формы `(1, 5)` и `history_emb_batch` формы `(1, 5, 4)`.

In [ ]:
history = torch.tensor([101, 7, 42, 7, 15])  # (T,) — ID последних 5 товаров (товар 7 смотрели дважды)
history_emb = torch.rand(5, 4)  # (T, E) — у каждого из 5 товаров вектор длины 4

In [ ]:
history_batch = history.unsqueeze(0)  # (T,) → (1, T) = (1, 5)
history_emb_batch = history_emb.unsqueeze(0)  # (T, E) → (1, T, E) = (1, 5, 4)

In [ ]:
need_tensor(history_batch, "history_batch")
need_tensor(history_emb_batch, "history_emb_batch")
assert history_batch.shape == (1, 5), f"форма {tuple(history_batch.shape)}, а нужна (1, 5)"
assert history_emb_batch.shape == (1, 5, 4), f"форма {tuple(history_emb_batch.shape)}, а нужна (1, 5, 4)"
assert torch.equal(history_batch[0], history), "внутри батча должна лежать та же история — числа не меняются"
assert torch.equal(history_emb_batch[0], history_emb)
print("8.1 — верно")

**8.2** Два способа собрать тензоры:
- `torch.stack` — создаёт **новую** ось;
- `torch.cat` — склеивает по **существующей**.

Даны `u1` и `u2` формы (3,). Получи `st` формы (2, 3) и `ct` формы (6,).

In [ ]:
u1 = torch.tensor([1, 2, 3])
u2 = torch.tensor([4, 5, 6])

In [ ]:
st = torch.stack([u1, u2])  # (2, 3)
ct = torch.cat([u1, u2])  # (6,)

In [ ]:
need_tensor(st, "st"); need_tensor(ct, "ct")
assert st.shape == (2, 3) and st.tolist() == [[1, 2, 3], [4, 5, 6]]
assert ct.shape == (6,) and ct.tolist() == [1, 2, 3, 4, 5, 6]
print("8.2 — верно")

**8.3** `permute` меняет порядок осей. Картинки часто хранят как (H, W, C), а PyTorch ждёт (C, H, W). Получи `chw` из `hwc`.

In [ ]:
hwc = torch.rand(4, 5, 3)  # (H, W, C)

In [ ]:
chw = hwc.permute(2, 0, 1)  # (C, H, W)

In [ ]:
need_tensor(chw, "chw")
assert chw.shape == (3, 4, 5)
assert torch.equal(chw[0], hwc[:, :, 0]), "первый канал должен совпасть с hwc[:, :, 0]"
print("8.3 — верно")

## 9. Итоговая задача: батч историй пользователей (рексис)

`H` — батч историй `(B, T, E)` = (8, 5, 4), `item` — эмбеддинг товара-кандидата `(E,)`.

1. `user_emb` — среднее эмбеддингов истории каждого пользователя, (8, 4).
2. `user_emb_k` — то же с `keepdim=True`, (8, 1, 4).
3. `H_centered` — из каждого товара истории вычесть эмбеддинг его пользователя, (8, 5, 4).
4. `H_flat` — каждую историю из `H_centered` вытянуть в вектор, (8, 20).
5. `scores` — скалярное произведение `user_emb` и `item`, (8,).
6. `best_user` — номер пользователя с самым большим скором, обычным числом Python.

In [ ]:
torch.manual_seed(1)
H = torch.rand(8, 5, 4)  # (B, T, E) — 8 пользователей × 5 товаров истории × эмбеддинг длины 4
item = torch.rand(4)  # (E,) — эмбеддинг товара-кандидата

In [ ]:
user_emb = H.mean(dim=1)  # (8, 5, 4) → свернули ось T → (8, 4)
user_emb_k = H.mean(dim=1, keepdim=True)  # (8, 1, 4) — ось T осталась длиной 1
H_centered = H - user_emb_k  # (8, 5, 4) - (8, 1, 4) → вектор пользователя растянулся на 5 товаров
H_flat = H_centered.view(8, -1)  # (8, 20) — 5 товаров × 4 числа в один вектор
scores = (user_emb * item).sum(dim=-1)  # (8, 4) * (4,) → (8, 4) → сумма по E → (8,)
best_user = scores.argmax().item()  # номер самого большого скора → обычное int

In [ ]:
for t, name in [
    (user_emb, "user_emb"),
    (user_emb_k, "user_emb_k"),
    (H_centered, "H_centered"),
    (H_flat, "H_flat"),
    (scores, "scores"),
]:
    need_tensor(t, name)

assert user_emb.shape == (8, 4), f"форма {tuple(user_emb.shape)}, а нужна (8, 4): сворачиваем ось T"
assert torch.allclose(user_emb, H.mean(dim=1)), "user_emb — среднее по истории каждого пользователя"
assert user_emb_k.shape == (8, 1, 4), f"форма {tuple(user_emb_k.shape)}, а нужна (8, 1, 4)"
assert H_centered.shape == H.shape
assert torch.allclose(
    H_centered.mean(dim=1), torch.zeros(8, 4), atol=1e-6
), "у каждого пользователя среднее по истории должно стать 0"
assert H_flat.shape == (8, 20), f"форма {tuple(H_flat.shape)}, а нужна (8, 20)"
assert torch.equal(H_flat[0], H_centered[0].reshape(-1)), "вытягивать нужно H_centered"
assert scores.shape == (8,), f"форма {tuple(scores.shape)}, а нужна (8,) — по скору на пользователя"
assert torch.allclose(scores, user_emb @ item), "скор = скалярное произведение user_emb и item (см. теорию)"
assert isinstance(best_user, int), f"best_user — {type(best_user).__name__}, а нужно обычное число (int)"
assert best_user == int(scores.argmax()), "это не пользователь с самым большим скором"
print("9 — верно. Блок пройден. Лучший пользователь для item:", best_user)

## Что запомнить из видео

| # | Правило |
|:---|:---|
| 1 | `<span style="color:#E6C07B">torch.tensor</span>(список, dtype=..., requires_grad=..., device=...)` |
| 2 | <span style="color:#C9A0DC">requires_grad=True</span> — только для float |
| 3 | `shape` == `size()`, `ndim` — число осей |
| 4 | индексация даёт <span style="color:#7CDBA0">тензор</span>, число — через <span style="color:#C9A0DC">item()</span> (например, для loss) |
| 5 | <span style="color:#C9A0DC">view</span> / <span style="color:#C9A0DC">reshape</span> меняют форму; `-1` — посчитай сам; если сомневаешься — <span style="color:#C9A0DC">reshape</span> |
| 6 | <span style="color:#C9A0DC">unsqueeze</span> добавляет ось 1, `squeeze` убирает |
| 7 | `dim` — ось, которая исчезает; `keepdim=True` оставляет её с размером 1 |
| 8 | `device = "cuda" if <span style="color:#E6C07B">torch.cuda.is_available</span>() else "cpu"`, потом `.to(device)` |
| 9 | тензоры для одной операции — на одном устройстве |
| 10 | <span style="color:#C9A0DC">detach()</span> — отцепить от графа градиентов |